# TFM · Notebook 01 — Datos, calidad y análisis exploratorio

**Objetivo de este notebook (Fases 2 y 3 del plan):**

1. Cargar los datos originales de UCI (`processed.cleveland.data`).
2. Comprobar la calidad: valores perdidos, duplicados y rangos.
3. Documentar que el `heart.csv` del máster tenía 6 valores imputados.
4. Crear la variable respuesta binaria y recodificar `restecg`.
5. Construir la **Tabla 1** (descriptiva por desenlace) y las figuras del EDA.
6. Guardar un fichero limpio para el notebook 02.

**Regla de oro:** este notebook *describe*; no elige variables ni toca el modelo.
Aquí todavía **no** se imputa nada: la imputación se hará dentro del `Pipeline`
en el notebook 02, ajustada solo con el 80 % de entrenamiento.

**Cómo usarlo en Google Colab:** `Entorno de ejecución → Ejecutar todas`.
Cuando lo pida, sube `processed.cleveland.data` (y, si quieres la comprobación del paso 3, también `heart.csv`).

## 1. Librerías

In [ ]:
# pandas: tablas de datos | numpy: cálculos | scipy: pruebas estadísticas | matplotlib: gráficos
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SEMILLA = 42                       # misma semilla en todo el TFM (reproducibilidad)
os.makedirs("figures", exist_ok=True)  # carpeta para guardar las figuras de la memoria

## 2. Carga de los datos originales

In [ ]:
# En Colab aparece un botón para subir archivos. Fuera de Colab, deja los ficheros
# en la misma carpeta que el notebook y no se pedirá nada.
try:
    from google.colab import files
    if not os.path.exists("processed.cleveland.data"):
        print("Sube processed.cleveland.data (y opcionalmente heart.csv):")
        files.upload()
except ImportError:
    pass  # no estamos en Colab

COLUMNAS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

# El fichero no tiene cabecera y marca los perdidos con "?"
df = pd.read_csv("processed.cleveland.data", header=None,
                 names=COLUMNAS, na_values="?")
print("Filas y columnas:", df.shape)
df.head()

### Diccionario de variables

| Variable | Significado | Tipo | Unidad / códigos |
|---|---|---|---|
| age | Edad | Continua | años |
| sex | Sexo | Binaria | 0 = mujer, 1 = hombre |
| cp | Tipo de dolor torácico | Nominal | 1 = angina típica, 2 = atípica, 3 = no anginoso, 4 = asintomático |
| trestbps | PA sistólica en reposo | Continua | mmHg |
| chol | Colesterol sérico | Continua | mg/dl |
| fbs | Glucemia en ayunas > 120 mg/dl | Binaria | 0 = no, 1 = sí |
| restecg | ECG en reposo | Nominal | 0 = normal, 1 = anomalía ST-T, 2 = HVI |
| thalach | FC máxima alcanzada | Continua | lpm |
| exang | Angina inducida por ejercicio | Binaria | 0 = no, 1 = sí |
| oldpeak | Descenso del ST con el ejercicio | Continua | mm |
| slope | Pendiente del ST en el pico | Nominal | 1 = ascendente, 2 = plana, 3 = descendente |
| ca | Vasos principales coloreados por fluoroscopia | Discreta | 0–3 |
| thal | Gammagrafía con talio | Nominal | 3 = normal, 6 = defecto fijo, 7 = defecto reversible |
| num | Diagnóstico angiográfico | Ordinal | 0 = < 50 % de estenosis; 1–4 = > 50 % |

Fuente: ficha del dataset en UCI (Janosi et al., 1989) y fichero `heart-disease.names`.

## 3. Calidad de los datos

In [ ]:
print("Valores perdidos por variable:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nFilas con algún perdido:", df.isna().any(axis=1).sum())
print("Filas duplicadas:", df.duplicated().sum())
df[df.isna().any(axis=1)]

In [ ]:
# Rangos: mínimos y máximos para detectar valores imposibles (no solo extremos).
df.describe().T[["min", "max", "mean"]].round(1)

### 3.1 Comprobación: el `heart.csv` del máster venía imputado

Esta celda solo se ejecuta si has subido también `heart.csv`. Compara los dos ficheros celda a celda.

In [ ]:
if os.path.exists("heart.csv"):
    heart = pd.read_csv("heart.csv")
    distintas = (df.fillna(-1).values != heart.values)
    filas = np.where(distintas.any(axis=1))[0]
    print("Celdas distintas:", distintas.sum(), "| filas (índice desde 0):", filas.tolist())
    comparacion = pd.concat({"UCI original": df.loc[filas, ["ca", "thal"]],
                             "heart.csv": heart.loc[filas, ["ca", "thal"]]}, axis=1)
    display(comparacion)
else:
    print("heart.csv no está disponible: se omite la comparación.")

**Para la memoria (métodos):** el fichero proporcionado por el máster contenía 6 valores
perdidos (4 en `ca`, 2 en `thal`) ya sustituidos por la moda, antes de cualquier partición.
Se partió del fichero original de UCI y la imputación se realizó dentro del pipeline de modelado.

## 4. Variable respuesta y recodificaciones

In [ ]:
# Desenlace binario: enfermedad coronaria angiográfica (estenosis > 50 %) = num >= 1
df["enfermedad"] = (df["num"] >= 1).astype(int)
print(df["num"].value_counts().sort_index())
print("\nEventos:", df["enfermedad"].sum(), "de", len(df),
      f"({df['enfermedad'].mean():.1%})")

In [ ]:
# restecg: la categoría 1 solo tiene 4 pacientes -> normal (0) frente a anormal (1 o 2).
# Es una regla fija, no aprende de los datos, así que puede hacerse antes de la partición.
print(df["restecg"].value_counts().sort_index())
df["restecg_anormal"] = (df["restecg"] >= 1).astype(int)

## 5. Tabla 1 — características de la cohorte según el desenlace

In [ ]:
CONTINUAS = {"age": "Edad, años", "trestbps": "PA sistólica, mmHg",
             "chol": "Colesterol, mg/dl", "thalach": "FC máxima, lpm",
             "oldpeak": "Descenso ST, mm"}
CATEGORICAS = {
    "sex": ("Sexo", {0: "Mujer", 1: "Hombre"}),
    "cp": ("Tipo de dolor", {1: "Angina típica", 2: "Angina atípica",
                             3: "Dolor no anginoso", 4: "Asintomático"}),
    "fbs": ("Glucemia > 120 mg/dl", {0: "No", 1: "Sí"}),
    "restecg_anormal": ("ECG en reposo", {0: "Normal", 1: "Anormal"}),
    "exang": ("Angina con el ejercicio", {0: "No", 1: "Sí"}),
    "slope": ("Pendiente del ST", {1: "Ascendente", 2: "Plana", 3: "Descendente"}),
    "ca": ("Vasos coloreados", {0: "0", 1: "1", 2: "2", 3: "3"}),
    "thal": ("Talio", {3: "Normal", 6: "Defecto fijo", 7: "Defecto reversible"}),
}

def tabla_1(datos, grupo, etiquetas_grupo):
    # Continuas: media ± DE si son aproximadamente normales (Shapiro p > 0,05 en ambos grupos),
    # si no, mediana [RIC]. Categóricas: n (%), con % por columna. Se informan los perdidos.
    g0, g1 = [datos[datos[grupo] == k] for k in (0, 1)]
    filas = []
    for var, nombre in CONTINUAS.items():
        a, b, tot = g0[var].dropna(), g1[var].dropna(), datos[var].dropna()
        normal = stats.shapiro(a).pvalue > 0.05 and stats.shapiro(b).pvalue > 0.05
        if normal:
            f = lambda x: f"{x.mean():.1f} ± {x.std():.1f}"
            p = stats.ttest_ind(a, b, equal_var=False).pvalue
        else:
            f = lambda x: f"{x.median():.1f} [{x.quantile(.25):.1f}–{x.quantile(.75):.1f}]"
            p = stats.mannwhitneyu(a, b).pvalue
        filas.append([nombre, f(tot), f(a), f(b), p])
    for var, (nombre, codigos) in CATEGORICAS.items():
        tabla = pd.crosstab(datos[var], datos[grupo])
        p = stats.chi2_contingency(tabla).pvalue if tabla.shape[0] > 1 else np.nan
        filas.append([nombre, "", "", "", p])
        for cod, txt in codigos.items():
            n_tot = (datos[var] == cod).sum()
            n0, n1 = (g0[var] == cod).sum(), (g1[var] == cod).sum()
            filas.append([f"   {txt}", f"{n_tot} ({n_tot/datos[var].notna().sum():.0%})",
                          f"{n0} ({n0/g0[var].notna().sum():.0%})",
                          f"{n1} ({n1/g1[var].notna().sum():.0%})", np.nan])
        if datos[var].isna().any():
            filas.append(["   Perdidos", str(datos[var].isna().sum()),
                          str(g0[var].isna().sum()), str(g1[var].isna().sum()), np.nan])
    cab = ["Variable", f"Total (n = {len(datos)})",
           f"{etiquetas_grupo[0]} (n = {len(g0)})", f"{etiquetas_grupo[1]} (n = {len(g1)})", "p"]
    t = pd.DataFrame(filas, columns=cab)
    t["p"] = t["p"].apply(lambda x: "" if pd.isna(x) else ("< 0.001" if x < 0.001 else f"{x:.3f}"))
    # Coma decimal, como corresponde a un texto en español
    for col in t.columns[1:]:
        t[col] = t[col].str.replace(".", ",", regex=False)
    return t

t1 = tabla_1(df, "enfermedad", ("Sin enfermedad", "Con enfermedad"))
t1.to_csv("figures/tabla1_por_desenlace.csv", index=False)
t1

**Cómo leerla:** las p-valores solo *describen* diferencias entre grupos; **no** se usan
para elegir variables (eso sería fuga de información). Pruebas usadas: t de Welch o
U de Mann-Whitney para continuas y χ² para categóricas. Con celdas pequeñas (`restecg`,
`slope` descendente), el χ² es aproximado: decláralo como limitación.

### 5.1 Tabla 1 por sexo (anexo)

In [ ]:
t1_sexo = tabla_1(df, "sex", ("Mujeres", "Hombres"))
# La propia variable de agrupación no se describe: se quitan sus tres filas
t1_sexo = t1_sexo[~t1_sexo["Variable"].str.strip().isin(["Sexo", "Mujer", "Hombre"])]
t1_sexo.to_csv("figures/tabla1_por_sexo.csv", index=False)
t1_sexo

## 6. El hallazgo del tipo de dolor torácico

In [ ]:
# Prevalencia de enfermedad por tipo de dolor: en Diamond y Forrester (1979) la angina típica
# tiene la mayor probabilidad. Aquí ocurre lo contrario: es un resultado a discutir.
dolor = df.groupby("cp")["enfermedad"].agg(n="count", con_enfermedad="sum", prevalencia="mean")
dolor.index = dolor.index.map(CATEGORICAS["cp"][1])
dolor["prevalencia"] = (dolor["prevalencia"] * 100).round(1)
dolor

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(dolor.index, dolor["prevalencia"], color="#4C72B0")
for i, (v, n) in enumerate(zip(dolor["prevalencia"], dolor["n"])):
    ax.text(i, v + 1.5, f"{v:.0f} %\n(n = {n})", ha="center", fontsize=9)
ax.set_ylabel("Pacientes con enfermedad (%)")
ax.set_ylim(0, 100)
ax.set_title("Prevalencia de enfermedad coronaria según el tipo de dolor")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_prevalencia_por_dolor.png", dpi=300)
plt.show()

## 7. Distribución de las variables continuas según el desenlace

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (var, nombre) in zip(axes, CONTINUAS.items()):
    datos = [df.loc[df["enfermedad"] == k, var].dropna() for k in (0, 1)]
    ax.boxplot(datos, widths=0.6)
    ax.set_xticks([1, 2], ["Sin enfermedad", "Con enfermedad"], fontsize=8)
    ax.set_title(nombre, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Variables continuas según enfermedad coronaria", y=1.02)
plt.tight_layout()
plt.savefig("figures/fig_continuas_por_desenlace.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Correlación entre variables continuas (Spearman)

In [ ]:
# Sirve para detectar redundancias (colinealidad), no para seleccionar variables.
corr = df[list(CONTINUAS) + ["ca"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
etiquetas = [CONTINUAS.get(c, "Vasos coloreados").split(",")[0] for c in corr.columns]
ax.set_xticks(range(len(corr)), etiquetas, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), etiquetas)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="ρ de Spearman")
plt.tight_layout()
plt.savefig("figures/fig_correlacion_spearman.png", dpi=300)
plt.show()

## 9. Guardar los datos para el notebook 02

In [ ]:
# Se guardan CON los perdidos: la imputación se hará en el notebook 02, dentro del Pipeline.
df.to_csv("datos_cleveland_limpios.csv", index=False)
print("Guardado datos_cleveland_limpios.csv:", df.shape)
try:
    from google.colab import files
    files.download("datos_cleveland_limpios.csv")
except ImportError:
    pass

## 10. Qué llevar a la memoria

- **Métodos:** fuente de datos (UCI, Cleveland Clinic), 303 pacientes, definición del desenlace
  (`num` ≥ 1), los 6 perdidos y su origen, la recodificación de `restecg`, y las pruebas
  estadísticas de la Tabla 1.
- **Resultados:** Tabla 1 (por desenlace), figura de prevalencia por tipo de dolor y figura
  de variables continuas. La Tabla 1 por sexo y la matriz de correlación, a anexos.
- **Discusión:** la relación inversa entre tipo de dolor y enfermedad, y por qué anticipa
  el mal rendimiento de la tabla de Diamond y Forrester en esta población remitida.